# Guangzhou Landsat Surface Temp Calculation

In [ ]:
import ee
import geemap
import os
import rasterio
from rasterio.merge import merge

# Initialize Earth Engine
try:
    ee.Initialize(project='applied-spatial-rotterdam')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='applied-spatial-rotterdam')



In [22]:
# 1. Define the complete broad envelope coordinates
lon_min, lon_max = 112.8500, 114.1500
lat_min, lat_max = 22.5000, 23.9500

# Split the region into a 2x2 grid matrix (4 manageable tiles)
lon_mid = (lon_min + lon_max) / 2
lat_mid = (lat_min + lat_max) / 2

tile_coords = [
    ("south_west", [lon_min, lat_min, lon_mid, lat_mid]),
    ("south_east", [lon_mid, lat_min, lon_max, lat_mid]),
    ("north_west", [lon_min, lat_mid, lon_mid, lat_max]),
    ("north_east", [lon_mid, lat_mid, lon_max, lat_max])
]

# Summer windows for 2023, 2024, 2025 — these get merged into ONE
# collection below, so the median/normalization is computed across
# all three summers at once (a single composite), not three separate ones.
SUMMER_DATE_RANGES = [
    ('2023-06-01', '2023-08-31'),
    ('2024-06-01', '2024-08-31'),
    ('2025-06-01', '2025-08-31'),
]

# 2. Imagery Processing Engine
def mask_clouds(img):
    qa = img.select('QA_PIXEL')
    return img.updateMask(qa.bitwiseAnd(1 << 4).eq(0).And(qa.bitwiseAnd(1 << 3).eq(0)))

def add_indicators(img):
    opt = img.select('SR_B.*').multiply(0.0000275).add(-0.2)
    ndvi = opt.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')
    lst = img.select('ST_B10').multiply(0.00341802).add(149.0).subtract(273.15).rename('LST_Celsius')
    return img.addBands(opt, None, True).addBands(lst, None, True).addBands(ndvi)

def normalize_lst(image, aoi):
    """Subtract per-image spatial mean to produce LST anomaly (normalized LST).

    The mean only needs to be a single representative scalar, so it's computed
    at a coarser scale (200m) with bestEffort/tileScale fallbacks. Doing this
    at the native 30m scale, per-image, across a 3-summer collection is what
    was timing out on the larger/more cloud-free tiles (e.g. south_east) even
    though NDVI — which has no per-image reduceRegion — exported fine.
    """
    lst = image.select('LST_Celsius')
    image_mean = lst.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=aoi,
        scale=30,
        bestEffort=True,
        maxPixels=1e9,
    ).getNumber('LST_Celsius')
    return (
        lst.subtract(image_mean)
        .rename('LST_Normalized')
        .toFloat()
        .copyProperties(image, image.propertyNames())
    )

def get_multiyear_collection(aoi):
    """Build a single pooled collection spanning summer 2023, 2024, and 2025."""
    spatial_filter = ee.Filter.bounds(aoi)
    cloud_filter = ee.Filter.lt('CLOUD_COVER', 30)

    collection = ee.ImageCollection([])  # empty collection to merge into
    for start, end in SUMMER_DATE_RANGES:
        date_filter = ee.Filter.date(start, end)
        l8 = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
              .filter(spatial_filter).filter(date_filter).filter(cloud_filter))
        l9 = (ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
              .filter(spatial_filter).filter(date_filter).filter(cloud_filter))
        window_collection = l8.merge(l9).limit(5, 'CLOUD_COVER')
        collection = collection.merge(window_collection)

    return collection.map(mask_clouds).map(add_indicators)

def get_processed_composite(tile_box):
    aoi = ee.Geometry.Rectangle(tile_box)
    collection = get_multiyear_collection(aoi)

    # Normalized LST: per-image spatial-mean anomaly, then pooled median
    lst_normalized = (
        collection
        .map(lambda img: normalize_lst(img, aoi))
        .select('LST_Normalized')
        .median()
        .clip(aoi)
    )

    # Raw median composite for NDVI (and unnormalized LST, if ever needed)
    summer_composite = collection.median().clip(aoi)

    return summer_composite.addBands(lst_normalized)



In [23]:
# 3. Loop through tiles and download safely under the 50MB limit
os.makedirs('temp_tiles', exist_ok=True)
lst_tile_files = []
ndvi_tile_files = []

print("--- Downloading Sub-Tiles (Bypassing 50MB Cap) ---")
for name, box in tile_coords:
    print(f"Downloading tile: {name}...")
    tile_composite = get_processed_composite(box)
    aoi_geo = ee.Geometry.Rectangle(box)

    lst_file = f"../data/temp_tiles/lst_{name}.tif"
    ndvi_file = f"../data/temp_tiles/ndvi_{name}.tif"

    geemap.ee_export_image(
        tile_composite.select('LST_Normalized').toFloat(),
        filename=lst_file, scale=30, region=aoi_geo, file_per_band=False,
        timeout=600
    )
    geemap.ee_export_image(
        tile_composite.select('NDVI').toFloat(),
        filename=ndvi_file, scale=30, region=aoi_geo, file_per_band=False
    )

    # Fail loudly here rather than letting a missing tile surface later
    # as a confusing rasterio "No such file or directory" error during merge.
    for f in (lst_file, ndvi_file):
        if not os.path.exists(f) or os.path.getsize(f) == 0:
            raise RuntimeError(
                f"Export failed for tile '{name}': '{f}' was not created. "
                "This usually means the EE export silently failed (e.g. no "
                "scenes matched the filters for this tile, an EE quota/timeout "
                "error, or a network hiccup). Check Earth Engine task status "
                "or try re-running this tile individually."
            )

    lst_tile_files.append(lst_file)
    ndvi_tile_files.append(ndvi_file)

--- Downloading Sub-Tiles (Bypassing 50MB Cap) ---
Generating URL ...
Please wait ...
Data downloaded to C:\Users\artem\Documents\TUDelft\ARFW0501\report\data\temp_tiles\lst_south_west.tif
Generating URL ...
Please wait ...
Data downloaded to C:\Users\artem\Documents\TUDelft\ARFW0501\report\data\temp_tiles\ndvi_south_west.tif
Generating URL ...
Please wait ...
Data downloaded to C:\Users\artem\Documents\TUDelft\ARFW0501\report\data\temp_tiles\lst_south_east.tif
Generating URL ...
Please wait ...
Data downloaded to C:\Users\artem\Documents\TUDelft\ARFW0501\report\data\temp_tiles\ndvi_south_east.tif
Generating URL ...
Please wait ...
Data downloaded to C:\Users\artem\Documents\TUDelft\ARFW0501\report\data\temp_tiles\lst_north_west.tif
Generating URL ...
Please wait ...
Data downloaded to C:\Users\artem\Documents\TUDelft\ARFW0501\report\data\temp_tiles\ndvi_north_west.tif
Generating URL ...
Please wait ...
Data downloaded to C:\Users\artem\Documents\TUDelft\ARFW0501\report\data\temp_tiles

In [24]:
# 4. Local Mosaic Stitching Engine
print("\n--- Stitching Tiles Together Locally ---")
os.makedirs('../data/guangzhou', exist_ok=True)

def merge_and_save(file_list, output_path):
    src_files = [rasterio.open(f) for f in file_list]
    mosaic, out_trans = merge(src_files)
    out_meta = src_files[0].meta.copy()
    out_meta.update({
        "height": mosaic.shape[1], "width": mosaic.shape[2],
        "transform": out_trans, "crs": src_files[0].crs
    })
    with rasterio.open(output_path, "w", **out_meta) as dest:
        dest.write(mosaic)
    for src in src_files:
        src.close()

merge_and_save(lst_tile_files, '../data/guangzhou/guangzhou_lst_normalized_2023_2025.tif')
merge_and_save(ndvi_tile_files, '../data/guangzhou/guangzhou_ndvi_2023_2025.tif')
print("✔ Tiled stitching complete! Composite files (summers 2023–2025) saved to data/guangzhou/")


--- Stitching Tiles Together Locally ---
✔ Tiled stitching complete! Composite files (summers 2023–2025) saved to data/guangzhou/


In [25]:
import ee
import geemap
import os

print("--- Downloading Guangzhou LCZ Raster from WUDAPT/GEE ---")

# Define Guangzhou bounding box (same envelope as LST/NDVI tiles)
aoi = ee.Geometry.Rectangle([112.8500, 22.5000, 114.1500, 23.9500])

# WUDAPT Local Climate Zones — global 100m dataset on GEE
lcz_collection = ee.ImageCollection("RUB/RUBCLIM/LCZ/global_lcz_map/latest")

# Get the most recent annual composite and clip to Guangzhou
lcz_image = (
    lcz_collection
    .filterBounds(aoi)
    .sort("system:time_start", False)  # Most recent first
    .first()
    .select("LCZ_Filter")              # Use the filtered (cleaned) band, not raw
    .clip(aoi)
)

os.makedirs("../data/guangzhou", exist_ok=True)
output_path = "../data/guangzhou/guangzhou_lcz_2018.tif"

print("Exporting LCZ raster (100m resolution)...")
geemap.ee_export_image(
    lcz_image.toFloat(),
    filename=output_path,
    scale=100,           # WUDAPT native resolution
    region=aoi,
    file_per_band=False
)

print(f"✔ LCZ raster saved to {output_path}")

--- Downloading Guangzhou LCZ Raster from WUDAPT/GEE ---
Exporting LCZ raster (100m resolution)...
Generating URL ...
Please wait ...
Data downloaded to C:\Users\artem\Documents\TUDelft\ARFW0501\report\data\guangzhou\guangzhou_lcz_2018.tif
✔ LCZ raster saved to ../data/guangzhou/guangzhou_lcz_2018.tif


In [4]:
# %pip install osmnx

  Using cached osmnx-2.1.0-py3-none-any.whl.metadata (4.7 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
Using cached osmnx-2.1.0-py3-none-any.whl (104 kB)
Using cached networkx-3.6.1-py3-none-any.whl (2.1 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [osmnx]kx]

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [30]:
import geopandas as gpd
import osmnx as ox
import requests
import os
os.makedirs("../data/guangzhou", exist_ok=True)

# ============================================================
# FIX: Download proper Guangzhou subdistrict polygons from OSM
# (Replaces the original boundary-only guangzhou_admin.geojson
#  which contained only LineStrings with empty properties)
# ============================================================

print("Downloading Guangzhou admin boundaries from OSM...")

# Get the official Guangzhou city boundary as a clipping mask
guangzhou_boundary = ox.geocode_to_gdf("Guangzhou, Guangdong, China")
guangzhou_boundary = guangzhou_boundary.to_crs(epsg=4326)

# Download admin level 9 subdistricts (街道 / Jiedao level)
gdf_gz = ox.features_from_place(
    "Guangzhou, Guangdong, China",
    tags={"boundary": "administrative", "admin_level": "9"}
)

# Keep only polygon geometries
gdf_gz = gdf_gz[gdf_gz.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])]
gdf_gz = gdf_gz.reset_index(drop=True)
gdf_gz = gdf_gz.to_crs(epsg=4326)

# Clip strictly to Guangzhou city boundary to remove stray neighbouring districts
gdf_gz = gpd.clip(gdf_gz, guangzhou_boundary)

print(f"Found {len(gdf_gz)} subdistrict units within Guangzhou city boundary")

output = "../data/guangzhou/guangzhou_admin.geojson"
gdf_gz[['geometry', 'name']].to_file(output, driver="GeoJSON")
print(f"✔ Saved {len(gdf_gz)} polygon subdistricts to {output}")

Found 412 subdistrict units within Guangzhou city boundary
✔ Saved 412 polygon subdistricts to ../data/guangzhou/guangzhou_admin.geojson


In [31]:
import geopandas as gpd
from rasterstats import zonal_stats

print("\n--- Running Zonal Statistics ---")
geojson_path = "../data/guangzhou/guangzhou_admin.geojson"
output_path  = "../data/guangzhou/guangzhou_admin_enriched.geojson"

gdf = gpd.read_file(geojson_path)
gdf['geometry'] = gdf['geometry'].make_valid()

# Extract values from our seamlessly stitched mosaics
gdf['mean_LST_celsius'] = [x['mean'] for x in zonal_stats(gdf, "../data/guangzhou/guangzhou_lst_normalized_2023_2025.tif", stats=['mean'])]
gdf['mean_NDVI']        = [x['mean'] for x in zonal_stats(gdf, "../data/guangzhou/guangzhou_ndvi_2023_2025.tif", stats=['mean'])]
gdf['majority_LCZ']     = [x['majority'] for x in zonal_stats(gdf, "../data/guangzhou/guangzhou_lcz_2018.tif", stats=['majority'])]

print(f"Missing LST values remaining: {gdf['mean_LST_celsius'].isna().sum()}")
print(f"Missing NDVI values remaining: {gdf['mean_NDVI'].isna().sum()}")

gdf.to_file(output_path, driver="GeoJSON")
print("✔ Enriched dataset saved successfully!")


--- Running Zonal Statistics ---
Missing LST values remaining: 0
Missing NDVI values remaining: 0
✔ Enriched dataset saved successfully!


In [33]:
import geopandas as gpd

print("--- Final Cleanup: Dropping Edge Thermal Gaps & Water Artifacts ---")
enriched_path = "../data/guangzhou/guangzhou_admin_enriched.geojson"

# 1. Load the dataset we just exported
gdf = gpd.read_file(enriched_path)
initial_rows = len(gdf)

# 2. Drop rows missing thermal data
gdf_clean = gdf.dropna(subset=['mean_LST_celsius', 'mean_NDVI'])

# 3. Drop negative NDVI values (water bodies / cloud artifacts)
gdf_clean = gdf_clean[gdf_clean['mean_NDVI'] >= 0]

final_rows = len(gdf_clean)

print(f"-> Total districts before cleanup: {initial_rows}")
print(f"-> Total districts kept with full coverage: {final_rows}")
print(f"-> Successfully removed {initial_rows - final_rows} boundary gap / water rows.")

# 4. Overwrite the GeoJSON with the finalized, clean dataset
gdf_clean.to_file(enriched_path, driver="GeoJSON")
print("✔ Cleaned dataset safely saved! Ready for R clustering.")

--- Final Cleanup: Dropping Edge Thermal Gaps & Water Artifacts ---
-> Total districts before cleanup: 412
-> Total districts kept with full coverage: 389
-> Successfully removed 23 boundary gap / water rows.
✔ Cleaned dataset safely saved! Ready for R clustering.
